# 📖 Notebook 1: Threat Modeling with STRIDE

Before you write a single line of code, you should ask: **"What could go wrong?"**

That's what threat modeling is — a structured way to identify security risks in your system **before** attackers find them.

At Microsoft, threat modeling is **mandatory** for every product under the SDL (Security Development Lifecycle). It's done during the **design phase**, before implementation begins.

## Learning Objectives

By the end of this notebook, you'll understand:
- What threat modeling is and why every team should do it
- The STRIDE framework for categorizing threats, and why it is applied **per DFD element type**
- How to draw a Data Flow Diagram (DFD) and put the trust boundaries in the right places
- How to derive severity from a likelihood x impact matrix instead of guessing it
- How to identify attack surfaces
- How to build a threat model for our Flask web app

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 08-enterprise/security-review
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
# We'll use these to interact with our Flask app and database
import requests
import json

BASE_URL = "http://localhost:5001"

# Verify the Flask app is running
try:
    resp = requests.get(f"{BASE_URL}/health")
    print(f"✅ Flask app is running: {resp.json()}")
except requests.ConnectionError:
    print("❌ Flask app is not running. Run: docker compose up -d")

## What is Threat Modeling?

**Threat modeling** is the process of identifying potential security threats to your system, then designing countermeasures.

Think of it like this: before building a house, an architect checks:
- Is the area prone to flooding? → Build on higher ground
- Are there earthquakes? → Reinforce the foundation
- Is the neighborhood safe? → Add locks and alarms

In software, we do the same thing:
- Can someone steal user credentials? → Add encryption + MFA
- Can someone inject malicious SQL? → Use parameterized queries
- Can someone access internal services? → Add network segmentation

### The Four Questions of Threat Modeling

Microsoft's SDL uses these four questions:

| # | Question | Example Answer |
|---|----------|----------------|
| 1 | What are we building? | A web app with user login, product search, comments |
| 2 | What can go wrong? | SQL injection, stolen passwords, XSS attacks |
| 3 | What are we doing about it? | Parameterized queries, bcrypt hashing, input escaping |
| 4 | Did we do a good job? | Security tests pass, pen test finds no critical issues |

## The STRIDE Framework

**STRIDE** is a mnemonic developed at Microsoft for categorizing security threats. Each letter represents a different type of attack:

| Letter | Threat | What It Means | Real-World Example |
|--------|--------|---------------|--------------------|
| **S** | Spoofing | Pretending to be someone else | Using stolen credentials to log in |
| **T** | Tampering | Modifying data you shouldn't | Changing the price of an item in a request |
| **R** | Repudiation | Denying you did something | Claiming you never placed that order |
| **I** | Information Disclosure | Exposing data that should be private | Leaking user emails through an API |
| **D** | Denial of Service | Making the system unavailable | Flooding the login endpoint with requests |
| **E** | Elevation of Privilege | Gaining access you shouldn't have | Regular user accessing admin endpoints |

### STRIDE Maps to Security Properties

Each STRIDE category is the **opposite** of a security property:

| Threat | Security Property |
|--------|-------------------|
| Spoofing | **Authentication** — verify identity |
| Tampering | **Integrity** — data hasn't been modified |
| Repudiation | **Non-repudiation** — actions are logged |
| Info Disclosure | **Confidentiality** — data is protected |
| Denial of Service | **Availability** — system stays up |
| Elevation of Privilege | **Authorization** — proper access control |

## Step 1: Draw the Data Flow Diagram (DFD)

The first step in threat modeling is understanding **how data flows** through your system.

A Data Flow Diagram uses four elements:

| Symbol | Element | Meaning |
|--------|---------|---------|
| Rectangle | External Entity | Users, third-party services (outside your control) |
| Circle | Process | Your code that transforms data |
| Parallel Lines | Data Store | Databases, file systems, caches |
| Arrow | Data Flow | Data moving between elements |

A DFD is only useful once you draw the **trust boundaries** on it — the lines where
the level of trust changes. Every arrow that crosses one is a place where data must
be re-validated, and almost every interesting threat lives on one of those crossings.

Here is the DFD for our Flask app, with the boundaries drawn in:

```
                    ZONE A — UNTRUSTED (the public internet)

   ┌──────────┐                                ┌──────────────────────┐
   │ Browser  │                                │ External web servers │
   │ (User)   │                                │ (user-supplied URLs) │
   └────┬─────┘                                └──────────▲───────────┘
        │ HTTP request / response                         │ outbound HTTP
        │                                                 │
 ═══════╪══════ TB-1 ═════════════════════════ TB-3 ══════╪═════════════════
        │  every byte arriving here is           the destination AND the
        │  attacker-controlled                   reply are attacker-chosen
        ▼                                                 │
   ┌────────────────────────────────────────────┬─────────┘
   │ (P) Flask App — port 5001                  │
   │     /api/login          /api/transfer      │
   │     /api/products/search  /api/fetch-url   │
   │     /comments/<product_id>                 │
   └──────────────────────┬─────────────────────┘
                          │ SQL queries / cache ops
 ═════════════════════════╪══════ TB-2 ══════════════════════════════════════
   Redis has no password,  │  the app's DB role owns every table: anything
   one DB role for all     │  that reaches this network reads everything
                          ▼
   ══════════════════════════      ══════════════════════════
    (DS) PostgreSQL                 (DS) Redis
    users, products,                sessions, rate limits,
    comments, audit_log             CSRF tokens
   ══════════════════════════      ══════════════════════════

                    ZONE B — OUR APPLICATION (compose network)
```

### Reading the trust boundaries

**TB-1 — internet → Flask.** The classic one. Every query parameter, JSON field,
header and cookie arriving here is attacker-controlled, and this is where the
app's whole attack surface is exposed. Notebook 2 attacks this boundary four
different ways.

**TB-2 — Flask → PostgreSQL / Redis.** This is *not* a boundary between our code
and someone else's — Flask and both data stores are ours. It is a boundary because
**nothing on the far side authenticates anybody**: Redis runs with no
`requirepass`, and the app connects to Postgres with a role that owns every table.
So whatever privilege Flask has, the data plane hands over in full.

Note what this means for SQL injection: injection does not "break through" TB-2.
It smuggles attacker intent across TB-1 and then **reuses Flask's own legitimate
privileges** on the other side. That is why the fix lives at TB-1 (parameterize the
query) and the *containment* lives at TB-2 (least-privilege DB roles), and why you
want both.

**TB-3 — Flask → the internet, outbound.** Two things cross it and both are
attacker-controlled: the **destination** (so the app can be aimed back at Zone B —
that is SSRF) and the **response body** (untrusted bytes entering our zone). An
outbound boundary is easy to forget precisely because the arrow points away from you.

## Step 2: Identify Attack Surfaces

The **attack surface** is every point where an attacker can interact with your system. Smaller is better.

Let's enumerate the attack surface of our Flask app:

In [ ]:
# Let's map out the attack surface of our Flask app
# In a real threat model, you'd use a spreadsheet or tool like Microsoft Threat Modeling Tool

attack_surface = [
    {
        "endpoint": "/api/login",
        "method": "POST",
        "input": "username, password (JSON body)",
        "trust_level": "Unauthenticated",
        "data_accessed": "User credentials in Postgres",
        "risks": ["Brute force", "Credential stuffing", "Information disclosure"],
    },
    {
        "endpoint": "/api/products/search",
        "method": "GET",
        "input": "q (query parameter)",
        "trust_level": "Unauthenticated",
        "data_accessed": "Product data in Postgres",
        "risks": ["SQL injection", "Data exfiltration"],
    },
    {
        "endpoint": "/comments/<product_id>",
        "method": "GET",
        "input": "product_id (URL path)",
        "trust_level": "Unauthenticated",
        "data_accessed": "Comments (user-generated content)",
        "risks": ["XSS via stored comments", "Path traversal"],
    },
    {
        "endpoint": "/api/comments",
        "method": "POST",
        "input": "user_id, product_id, content (JSON body)",
        "trust_level": "Unauthenticated",
        "data_accessed": "Writes to comments table",
        "risks": ["XSS payload injection", "Spam", "Missing auth"],
    },
    {
        "endpoint": "/api/transfer",
        "method": "POST",
        "input": "from_user, to_user, amount (JSON body)",
        "trust_level": "Unauthenticated",
        "data_accessed": "Financial operation",
        "risks": ["CSRF", "Missing authentication", "Missing authorization"],
    },
    {
        "endpoint": "/api/fetch-url",
        "method": "GET",
        "input": "url (query parameter)",
        "trust_level": "Unauthenticated",
        "data_accessed": "Any URL (internal or external)",
        "risks": ["SSRF", "Internal network scanning", "Data exfiltration"],
    },
]

print("🎯 Attack Surface Analysis")
print("=" * 80)
for entry in attack_surface:
    print(f"\n📍 {entry['method']} {entry['endpoint']}")
    print(f"   Input: {entry['input']}")
    print(f"   Trust: {entry['trust_level']}")
    print(f"   Data:  {entry['data_accessed']}")
    print(f"   Risks: {', '.join(entry['risks'])}")

## Step 3: Apply STRIDE to Each Element

Now we apply STRIDE to each part of the data flow diagram. For each element, we ask:
**is this vulnerable to Spoofing? Tampering? Repudiation? Information Disclosure?
Denial of Service? Elevation of Privilege?**

### STRIDE-per-element: not every letter applies to every element

This is the part people usually get wrong. STRIDE is applied *per element type*, and
each type is only susceptible to a subset of the six categories:

| DFD element | S | T | R | I | D | E | Why |
|-------------|---|---|---|---|---|---|-----|
| **External entity** (browser, third party) | ✅ | — | ✅ | — | — | — | It has an identity to spoof and actions to deny, but it holds no data of ours to tamper with or leak |
| **Process** (our Flask app) | ✅ | ✅ | ✅ | ✅ | ✅ | ✅ | Code that runs with privileges is exposed to all six |
| **Data store** (Postgres, Redis) | — | ✅ | ✅ | ✅ | ✅ | — | A table cannot impersonate anyone or grant itself rights, but it can be altered, read, filled up, or have its logs erased |
| **Data flow** (an arrow) | — | ✅ | — | ✅ | ✅ | — | Bytes in transit can be modified, read, or blocked — a wire has no identity and no privileges |

A model that only lists threats against *processes* — which is the easy, obvious
half — has no coverage of its data stores or its wires. Ours starts out exactly that
way, so we will fix it below and then have the code check the coverage for us.

Let's build the threat model programmatically:

In [ ]:
# Building a threat model using STRIDE
# In practice, teams use tools like Microsoft Threat Modeling Tool or OWASP Threat Dragon

stride_categories = {
    "S": "Spoofing",
    "T": "Tampering",
    "R": "Repudiation",
    "I": "Information Disclosure",
    "D": "Denial of Service",
    "E": "Elevation of Privilege",
}

# STRIDE-per-element: which categories can even apply to which DFD element type.
# (This is the table from the markdown cell above, in code so we can check it.)
APPLICABLE = {
    "external entity": {"S", "R"},
    "process":         {"S", "T", "R", "I", "D", "E"},
    "data store":      {"T", "R", "I", "D"},
    "data flow":       {"T", "I", "D"},
}

# Severity is NOT hand-assigned — it is looked up in the likelihood x impact risk
# matrix drawn in Step 4 below. Writing it this way means a reviewer argues about
# two small judgements ("how likely?", "how bad?") instead of one vague one, and
# the labels stay internally consistent.
RISK_MATRIX = {
    ("High",   "Low"): "Medium", ("High",   "Medium"): "High",   ("High",   "High"): "Critical",
    ("Medium", "Low"): "Low",    ("Medium", "Medium"): "Medium", ("Medium", "High"): "High",
    ("Low",    "Low"): "Low",    ("Low",    "Medium"): "Low",    ("Low",    "High"): "Medium",
}


def risk(likelihood: str, impact: str) -> str:
    """Look up a severity in the risk matrix. Raises on an unknown pair."""
    return RISK_MATRIX[(likelihood, impact)]


threat_model = [
    # ================= PROCESS: the Flask app, endpoint by endpoint =================
    # --- Login Endpoint ---
    {
        "element": "/api/login", "element_type": "process", "stride": "S",
        "threat": "Attacker brute-forces passwords to impersonate a user",
        "likelihood": "High",   # no rate limit at all; fully scriptable
        "impact": "Medium",     # one account per success
        "mitigation": "Rate limiting (Redis counter), account lockout after 5 attempts",
    },
    {
        "element": "/api/login", "element_type": "process", "stride": "I",
        "threat": "Error message reveals whether username exists",
        "likelihood": "High",   # two requests and you know
        "impact": "Low",        # a username list, not a credential
        "mitigation": "Return same error, same status AND same timing for wrong username as for wrong password",
    },
    {
        "element": "/api/login", "element_type": "process", "stride": "R",
        "threat": "No audit log of login attempts — can't trace breaches",
        "likelihood": "Medium",
        "impact": "Medium",     # slows every future investigation
        "mitigation": "Log all login attempts (success and failure) to audit_log table",
    },
    # --- Product Search ---
    {
        "element": "/api/products/search", "element_type": "process", "stride": "T",
        "threat": "SQL injection modifies the query to extract or delete data",
        "likelihood": "High",   # unauthenticated, one query parameter
        "impact": "High",
        "mitigation": "Parameterized queries (never concatenate user input into SQL)",
    },
    {
        "element": "/api/products/search", "element_type": "process", "stride": "I",
        "threat": "SQL injection UNIONs the users table into the product results",
        "likelihood": "High",
        "impact": "High",       # every credential and email in one request
        "mitigation": "Parameterized queries + principle of least privilege on DB user",
    },
    # --- Comments ---
    {
        "element": "/comments/<product_id>", "element_type": "process", "stride": "T",
        "threat": "Stored XSS: a comment's script runs in every other viewer's browser",
        "likelihood": "High",
        "impact": "Medium",     # victim's session, not the server
        "mitigation": "HTML-escape all user content, Content-Security-Policy header",
    },
    # --- Transfer ---
    {
        "element": "/api/transfer", "element_type": "process", "stride": "S",
        "threat": "CSRF: a malicious site makes the victim's browser send the transfer, so the request carries the victim's identity",
        "likelihood": "High",
        "impact": "High",
        "mitigation": "CSRF tokens, SameSite cookies, Origin header validation",
    },
    {
        "element": "/api/transfer", "element_type": "process", "stride": "E",
        "threat": "No authentication required — an anonymous caller performs a privileged operation",
        "likelihood": "High",
        "impact": "High",
        "mitigation": "Require valid JWT token, verify user owns the source account",
    },
    # --- Fetch URL (SSRF) ---
    {
        "element": "/api/fetch-url", "element_type": "process", "stride": "I",
        "threat": "SSRF: attacker aims our own server back at Zone B (Redis, Postgres, cloud metadata)",
        "likelihood": "High",
        "impact": "High",
        "mitigation": "URL allowlist, block private IP ranges, HTTPS only, do not follow redirects",
    },
    {
        "element": "/api/fetch-url", "element_type": "process", "stride": "D",
        "threat": "Attacker makes server fetch huge files, exhausting memory",
        "likelihood": "Medium",
        "impact": "Medium",
        "mitigation": "Response size limits, timeouts, rate limiting",
    },

    # ============ DATA STORES AND DATA FLOWS: the half people forget ============
    {
        "element": "PostgreSQL", "element_type": "data store", "stride": "T",
        "threat": "The app's DB role can UPDATE and DELETE every table, so one injection rewrites prices or wipes rows",
        "likelihood": "Medium",  # needs an injection point first
        "impact": "High",
        "mitigation": "Separate least-privilege roles per operation; no DDL rights for the app role",
    },
    {
        "element": "PostgreSQL", "element_type": "data store", "stride": "R",
        "threat": "audit_log sits in the same database under the same role, so an attacker can delete the evidence",
        "likelihood": "Medium",
        "impact": "Medium",
        "mitigation": "Ship audit records off-box to append-only storage the app cannot rewrite",
    },
    {
        "element": "Redis", "element_type": "data store", "stride": "I",
        "threat": "Redis has no requirepass, so anything that reaches the compose network reads every session and CSRF token",
        "likelihood": "High",    # and TB-3 gives an attacker a way onto that network
        "impact": "High",
        "mitigation": "requirepass / Redis ACLs, bind to an internal interface, network policy",
    },
    {
        "element": "Browser -> Flask", "element_type": "data flow", "stride": "I",
        "threat": "The flow is plain HTTP on port 5001, so passwords and JWTs are readable by anyone on the path",
        "likelihood": "Medium",
        "impact": "High",
        "mitigation": "TLS everywhere + HSTS (see the security-headers section in notebook 2)",
    },
]

# Method check: a category that cannot apply to an element type is a modelling
# error, not a finding. Catch it here rather than in a review meeting.
for t in threat_model:
    allowed = APPLICABLE[t["element_type"]]
    assert t["stride"] in allowed, (
        f"{t['stride']} is not applicable to a {t['element_type']} "
        f"({t['element']}); allowed here: {sorted(allowed)}"
    )
    t["severity"] = risk(t["likelihood"], t["impact"])

print("🔍 STRIDE Threat Model for Security Demo App")
print("=" * 90)

for t in threat_model:
    category = stride_categories[t['stride']]
    print(f"\n🏷️  [{t['stride']}] {category} — {t['element']} ({t['element_type']})")
    print(f"   Threat:      {t['threat']}")
    print(f"   Likelihood:  {t['likelihood']}   Impact: {t['impact']}   -> Severity: {t['severity']}")
    print(f"   Mitigation:  {t['mitigation']}")

print(f"\n{len(threat_model)} threats across "
      f"{len({t['element'] for t in threat_model})} elements.")

## Step 4: Prioritize Threats

Not all threats are equal. We use a **risk matrix** to prioritize. Severity is not a
gut call — it is the cell you land in when you cross **how likely** with **how bad**:

```
                    Impact
              Low    Medium    High
         ┌─────────┬─────────┬─────────┐
  High   │ Medium  │  High   │Critical │  ← Likelihood
         ├─────────┼─────────┼─────────┤
  Medium │  Low    │ Medium  │  High   │
         ├─────────┼─────────┼─────────┤
  Low    │  Low    │  Low    │ Medium  │
         └─────────┴─────────┴─────────┘
```

This is exactly the `RISK_MATRIX` dict the previous cell used — the table above is
the documentation and the dict is the implementation, so a threat can never be
labelled "Critical" in the write-up and "Medium" in the tracker.

> ⚠️ **Be honest about what this is.** A risk matrix is a *communication* tool, not a
> measurement. "High likelihood" is still someone's opinion, and two reviewers will
> disagree. Its value is that it forces the disagreement into the open (*"you think
> that's Medium impact? why?"*) instead of hiding it inside a single unexplained
> severity label. The same caveat applies to DREAD, which averages five such
> opinions and gets a decimal number out — the decimal does not make it measurement.
> Use the matrix to sort a backlog; don't use it to claim precision you don't have.

At Microsoft, the SDL requires that **all Critical and High severity threats must be
mitigated before release**. Medium threats need a documented plan.

In [ ]:
# Count threats by severity and by STRIDE category, then check the model's coverage.
from collections import Counter, defaultdict

severity_counts = Counter(t["severity"] for t in threat_model)
stride_counts = Counter(stride_categories[t["stride"]] for t in threat_model)

print("📊 Threat Summary")
print("=" * 40)
print("\nBy Severity:")
for severity in ["Critical", "High", "Medium", "Low"]:
    count = severity_counts.get(severity, 0)
    bar = "█" * count
    print(f"  {severity:10s} {bar} ({count})")

print("\nBy STRIDE Category:")
for letter, name in stride_categories.items():
    count = stride_counts.get(name, 0)
    bar = "█" * count
    print(f"  [{letter}] {name:25s} {bar} ({count})")

# Critical + High must be fixed before release (per Microsoft SDL)
must_fix = severity_counts.get("Critical", 0) + severity_counts.get("High", 0)
print(f"\n🚨 Must fix before release: {must_fix} threats")
print(f"📋 Need documented plan:   {severity_counts.get('Medium', 0)} threats")

# ---------------------------------------------------------------------------
# Coverage: which (element, STRIDE category) pairs did we actually think about?
# A threat model's most dangerous output is the cell you never looked at, so
# print the gaps instead of quietly leaving them off the list.
# ---------------------------------------------------------------------------
modelled = defaultdict(set)
element_type = {}
element_order = []
for t in threat_model:
    if t["element"] not in element_type:
        element_order.append(t["element"])
        element_type[t["element"]] = t["element_type"]
    modelled[t["element"]].add(t["stride"])

letters = list(stride_categories)          # S T R I D E
header = "  ".join(letters)
print("\n🗺️  Coverage grid   ● modelled   · applicable, not yet modelled   — n/a")
print(f"  {'element':30s} {header}")
gaps = []
for element in element_order:
    row = []
    for letter in letters:
        if letter not in APPLICABLE[element_type[element]]:
            row.append("—")
        elif letter in modelled[element]:
            row.append("●")
        else:
            row.append("·")
            gaps.append((element, letter))
    row_str = "  ".join(row)
    print(f"  {element:30s} {row_str}")

print(f"\n  {len(gaps)} applicable cells are still empty. That is not a clean bill of")
print("  health — it is the list of questions this review has not asked yet, e.g.:")
for element, letter in gaps[:3]:
    print(f"    - can {element} be attacked via {stride_categories[letter]}?")

# --- Guardrails so this model can't quietly drift out of sync with the prose ---
assert sum(severity_counts.values()) == len(threat_model)
assert (severity_counts["Critical"], severity_counts["High"], severity_counts["Medium"]) \
    == (6, 4, 4), (
    f"severity mix changed to {dict(severity_counts)} — the threat-model document "
    f"in the next cell quotes these numbers, update it too"
)
assert must_fix == 10, f"expected 10 must-fix threats, got {must_fix}"
# Every element drawn on the DFD must appear in the model at least once.
assert set(element_order) >= {"PostgreSQL", "Redis", "Browser -> Flask"}, (
    "the data stores and data flows fell out of the model again — a process-only "
    "STRIDE model is the classic incomplete threat model"
)
# Every STRIDE letter must be exercised somewhere, or the framework is decoration.
assert all(stride_counts[name] > 0 for name in stride_categories.values()), (
    f"unused STRIDE categories: "
    f"{[n for n in stride_categories.values() if not stride_counts[n]]}"
)
print("\n✅ model self-checks passed")

## Step 5: Verify Threats with Real Attacks

A threat model that is never tested is a wish list. Let's verify one of our
identified threats is real — the `[I] Information Disclosure` entry that says the
login endpoint reveals whether a user exists.

Note that "same error message" is only half the fix. If the safe endpoint skips the
password hash when the username is unknown, it answers a *lot* faster for unknown
users, and the timing alone re-opens the enumeration. We check both channels below.

In [ ]:
# Verify threat: Information Disclosure on /api/login
# The vulnerable endpoint returns 404 "User not found" vs 401 "Invalid password".

import statistics
import time

# The SAFE endpoint rate-limits failures by IP (5 per 15 minutes). Re-running this
# notebook would trip that limit and answer 429, which looks like the demo broke.
# Clear the counters so the cell is re-runnable.
try:
    import redis
    _r = redis.Redis(host="localhost", port=6379, decode_responses=True)
    _r.ping()

    def reset_login_limits():
        stale = _r.keys("login_attempts:*")
        if stale:
            _r.delete(*stale)
        return True
except Exception as _e:  # redis not reachable from the notebook
    print(f"⚠️  could not reach Redis to clear rate-limit counters: {_e}")

    def reset_login_limits():
        return False


reset_login_limits()
print("Testing /api/login information disclosure...\n")

# Test with a username that EXISTS in the database
resp1 = requests.post(f"{BASE_URL}/api/login", json={
    "username": "alice",
    "password": "wrong-password"
})
print(f"Existing user, wrong password:")
print(f"  Status: {resp1.status_code}")
print(f"  Body:   {resp1.json()}")

# Test with a username that DOES NOT exist
resp2 = requests.post(f"{BASE_URL}/api/login", json={
    "username": "nonexistent_user_xyz",
    "password": "wrong-password"
})
print(f"\nNon-existent user:")
print(f"  Status: {resp2.status_code}")
print(f"  Body:   {resp2.json()}")

print("\n⚠️  Notice the difference!")
print("The vulnerable endpoint answers 401 'Invalid password' when the account is")
print("real and 404 'User not found' when it isn't. Two requests tell an attacker")
print("which usernames exist, before they try a single password.")

assert resp1.status_code != resp2.status_code, (
    f"the vulnerable endpoint stopped leaking user existence "
    f"({resp1.status_code} for both) — this notebook can no longer demonstrate "
    f"the [I] threat it claims to"
)

# Now test the SAFE endpoint
print("\n" + "=" * 60)
print("Testing /api/login/safe (fixed version)...\n")

reset_login_limits()
resp3 = requests.post(f"{BASE_URL}/api/login/safe", json={
    "username": "alice",
    "password": "wrong-password"
})
print(f"Existing user, wrong password:")
print(f"  Status: {resp3.status_code}")
print(f"  Body:   {resp3.json()}")

resp4 = requests.post(f"{BASE_URL}/api/login/safe", json={
    "username": "nonexistent_user_xyz",
    "password": "wrong-password"
})
print(f"\nNon-existent user:")
print(f"  Status: {resp4.status_code}")
print(f"  Body:   {resp4.json()}")

assert resp3.status_code == resp4.status_code == 401, (
    f"safe endpoint gave {resp3.status_code} vs {resp4.status_code}; expected 401 "
    f"for both (429 means the rate-limit counter was not cleared)"
)
assert resp3.json() == resp4.json(), (
    f"safe endpoint still leaks via the body: {resp3.json()} vs {resp4.json()}"
)
print("\n✅ The safe endpoint returns the same status and the same body in both cases.")


# --- The second channel: response time ---------------------------------------
def timed_login(username, samples=3):
    """Median response time for a failed login, with the rate limit reset first."""
    durations = []
    for _ in range(samples):
        if not reset_login_limits():
            return None
        start = time.perf_counter()
        requests.post(f"{BASE_URL}/api/login/safe",
                      json={"username": username, "password": "wrong-password"})
        durations.append(time.perf_counter() - start)
    return statistics.median(durations)


real_ms = timed_login("alice")
fake_ms = timed_login("nonexistent_user_xyz")
if real_ms is None or fake_ms is None:
    print("\n(skipped the timing check — Redis was not reachable)")
else:
    print(f"\n⏱️  median response time, real username:    {real_ms * 1000:6.1f} ms")
    print(f"⏱️  median response time, unknown username: {fake_ms * 1000:6.1f} ms")
    print("\nBoth paths run exactly one bcrypt comparison — the unknown-username path")
    print("checks the password against a dummy hash — so the times are comparable.")
    print("Without that dummy comparison the unknown-username path would return in")
    print("well under a millisecond and leak the same information the error message")
    print("no longer does.")
    print("\n(These are indicative numbers from a handful of samples over a local")
    print(" network, not a statistical result. A real timing-attack assessment needs")
    print(" thousands of samples and a distribution test — which is precisely why you")
    print(" fix the shape of the code rather than measuring your way to 'probably fine'.)")

reset_login_limits()  # leave the rate-limit counters clean for the next notebook

## Threat Model Document Template

At Microsoft and other large companies, threat models are documented formally. Here's
a template, filled in with the numbers the cells above computed (the assertion in the
counting cell pins those numbers, so the document and the model cannot drift apart):

```
# Threat Model: Security Demo Web Application

## 1. System Overview
- Flask web application serving product catalog
- PostgreSQL for data storage
- Redis for session management and rate limiting
- Exposed on port 5001

## 2. Trust Boundaries
- TB-1  Internet -> Flask App          (untrusted -> our zone; all input hostile)
- TB-2  Flask App -> PostgreSQL/Redis  (unauthenticated data plane; app privilege
                                        is total privilege on the far side)
- TB-3  Flask App -> External URLs     (outbound; destination AND response are
                                        attacker-controlled)

## 3. Data Classification
- HIGH: User passwords, JWT secrets, API keys
- MEDIUM: User email addresses, session tokens
- LOW: Product names, prices, public comments

## 4. STRIDE Analysis
- 14 threats across 8 DFD elements (5 processes, 2 data stores, 1 data flow)
- 6 Critical, 4 High, 4 Medium — severity derived from likelihood x impact
- All 10 Critical/High must be mitigated before launch
- Coverage is NOT complete: see the coverage grid for the applicable
  (element, category) pairs this review has not yet examined

## 5. Mitigations
- [See threat table above]

## 6. Sign-off
- Security team review: PENDING
- Pen test: PENDING
- SDL compliance: PENDING
```

Notice section 4 admits what the model does *not* cover. A threat model that reports
only findings reads like a clean bill of health; one that also reports its blind
spots tells the next reviewer where to start.

## Attack Surface Reduction

One of the most effective security measures is **reducing the attack surface** — removing features and endpoints that aren't needed.

### Checklist for Attack Surface Reduction

| Question | Action |
|----------|--------|
| Do all endpoints need to be public? | Add authentication where possible |
| Does the app need to fetch arbitrary URLs? | Remove or restrict `/api/fetch-url` |
| Does the app need debug mode in production? | Disable `debug=True` |
| Are all database columns necessary in API responses? | Return only needed fields |
| Are default credentials changed? | Always change defaults |
| Are unused ports closed? | Only expose what's needed |

In [ ]:
# Let's verify our app's attack surface by checking which endpoints exist.
# Everything here is local: no endpoint in this check depends on the public
# internet, so the result is the same on a plane as it is at your desk.

endpoints_to_check = [
    ("GET",  "/health"),
    ("GET",  "/api/products/search?q=laptop"),
    ("GET",  "/api/products/search/safe?q=laptop"),
    ("GET",  "/comments/1"),
    ("GET",  "/comments/1/safe"),
    ("POST", "/api/login"),
    ("POST", "/api/login/safe"),
    ("POST", "/api/transfer"),
    ("POST", "/api/transfer/safe"),
    ("GET",  "/api/fetch-url?url=http://localhost:5001/health"),
    ("GET",  "/api/fetch-url/safe?url=https://127.0.0.1/health"),
    ("GET",  "/api/audit-log"),
]

print("🎯 Endpoint Accessibility Check")
print("=" * 80)
print("We send every request with NO cookie, NO Authorization header, no identity")
print("of any kind, and record what the server says.\n")

statuses = {}
for method, path in endpoints_to_check:
    try:
        if method == "GET":
            resp = requests.get(f"{BASE_URL}{path}", timeout=5)
        else:
            resp = requests.post(f"{BASE_URL}{path}", json={}, timeout=5)
        statuses[(method, path)] = resp.status_code
        # A 403 here is a per-request check (CSRF token, URL allowlist), and the
        # 401 is a failed credential check on the login endpoint itself. Neither
        # is "you must be logged in to use this API".
        note = "  ← per-request check, not a session requirement" \
            if resp.status_code == 403 else ""
        icon = "✅" if resp.status_code < 500 else "❌"
        print(f"  {icon} {method:5s} {path:48s} → {resp.status_code}{note}")
    except Exception as e:
        statuses[(method, path)] = None
        print(f"  ❌ {method:5s} {path:48s} → Error: {e}")

answered = [s for s in statuses.values() if s is not None]
print(f"\n⚠️  {len(answered)}/{len(endpoints_to_check)} endpoints answered an anonymous request.")
print("Not one of them requires a session. The /safe variants do reject some calls")
print("(403), but that is a CSRF token check and a URL allowlist — per-request checks")
print("that say nothing about *who* is calling. `/api/audit-log`, the security log")
print("itself, hands its contents to a complete stranger. In a real app most of these")
print("should sit behind a valid session or JWT, which is the cheapest single")
print("reduction in attack surface available to this app.")

assert len(answered) == len(endpoints_to_check), (
    f"only {len(answered)} of {len(endpoints_to_check)} endpoints responded — "
    f"is docker compose up?"
)
# The claim above is "these serve data to an anonymous caller". Pin it.
for key in [("GET", "/api/audit-log"),
            ("GET", "/api/products/search?q=laptop"),
            ("GET", "/comments/1")]:
    assert statuses[key] == 200, (
        f"{key[1]} returned {statuses[key]} to an anonymous caller; this cell's "
        f"whole point is that it returns 200 with data"
    )

## 🔑 Key Takeaways

1. **Threat model BEFORE you code** — it's cheaper to fix design issues than implementation bugs
2. **Use STRIDE per element** — a data store cannot spoof anyone and a wire has no privileges, so only some letters apply. A model that only lists threats against endpoints has skipped its data stores and its data flows
3. **Draw a Data Flow Diagram** with the trust boundaries on it — a boundary is where trust *changes*, and the interesting threats all sit on a crossing. Don't forget the outbound one
4. **Minimize attack surface** — remove, restrict, or authenticate everything you can
5. **Derive severity, don't declare it** — likelihood x impact from a documented matrix, so two reviewers argue about two small judgements instead of one vague label. And say out loud that the matrix is a communication tool, not a measurement
6. **Publish your blind spots** — the coverage grid's empty cells are the questions the review has not asked yet. A model that reports only findings reads like a clean bill of health
7. **Verify with real tests** — don't just assume mitigations work

## ➡️ Next: Notebook 2 — Common Vulnerabilities

Now that we've identified threats, let's **exploit them** and then **fix them**.